# Corrected FADC3D encoder — seed 42

**Branch:** `feature/fadc3d-correct`

Discrete voxelwise 3D frequency-adaptive dilation with SHARED kernels.

- One learnable base kernel per adaptive convolution.
- Dilations exactly `(1,1,1) / (2,2,2) / (3,3,3)`, per-voxel `k_att` softmax over dim=1.
- Kernel-side AdaKern3D attention on the `W_low + W_high` decomposition.
- `FrequencySelection3D` on the input feature map, defaults `k_list=[2,4,8]`.
- Corrected FADC only in `enc1..enc4`; bottleneck + decoder use plain Conv3d.

Deliberately not claimed: continuous AdaDR, deformable Conv3d, or any Dice number.

**Hardened notebook** — every substantive cell aborts loudly (SystemExit) if
its prerequisite fails, so a stale checkout or failed test never proceeds to
100-epoch training silently.


In [ ]:
# ── CONFIG ─────────────────────────────────────────────────────────────
SEED                = 42
GIT_BRANCH          = "feature/fadc3d-correct"

DATA_ROOT              = "/kaggle/input/datasets/bharathvemurik/mama-mia-preprocessed-cache-2ch"
PREPROCESSED_CACHE_DIR = DATA_ROOT
CODE_DIR               = "/kaggle/working/FADC-3D"

OUTPUT_DIR_SMOKE  = "/kaggle/working/outputs/fadc3d_correct_encoder_smoke"
OUTPUT_DIR_FULL   = "/kaggle/working/outputs/fadc3d_correct_encoder_s42"

# Full-training hyperparameters (UNCHANGED — do NOT modify without a new run).
EPOCHS         = 100
BATCH_SIZE     = 2
NUM_WORKERS    = 4
PATCH_SIZE     = [128, 128, 64]
LEARNING_RATE  = 1e-4
WARMUP_EPOCHS  = 5
DEEP_SUPERVISION = True

# k_att schedule (UNCHANGED).
K_ATT_TEMP_START    = 2.0
K_ATT_TEMP_END      = 1.0
K_ATT_ANNEAL_EPOCHS = 60

# Attention diversity aux DISABLED for this run (UNCHANGED).
ATTN_DIVERSITY_WEIGHT = 0.0

# Smoke-test parameters.
SMOKE_PATCH_SIZE = [48, 48, 24]

MODEL_NAME = "unet3d_fadc_encoder_correct"

# ── VALIDATION SCHEDULE (formal only) ──────────────────────────────────
# One protocol: all validation cases, sliding-window at overlap=0.5.
# Runs every VAL_EVERY epochs. Only formal validation updates
# best_model.pth. No fast/proxy variant.
VAL_EVERY           = 20     # formal validation every N epochs
VAL_OVERLAP         = 0.5    # canonical evaluation overlap
VAL_SW_BATCH_SIZE   = 4
CHECKPOINT_EVERY    = 10     # also save named checkpoint_epochNNN.pth

# ── OPTIONAL RESUME ────────────────────────────────────────────────────
# Leave empty for a FRESH training run. To resume after a Kaggle timeout:
# upload OUTPUT_DIR_FULL/last_checkpoint.pth as a Dataset, then set
# RESUME_FROM to the mount path here.
RESUME_FROM = ""

print(f"SEED                : {SEED}")
print(f"BRANCH              : {GIT_BRANCH}")
print(f"MODEL_NAME          : {MODEL_NAME}")
print(f"OUTPUT_DIR_FULL     : {OUTPUT_DIR_FULL}")
print(f"OUTPUT_DIR_SMOKE    : {OUTPUT_DIR_SMOKE}")
print(f"PATCH_SIZE (train)  : {PATCH_SIZE}")
print(f"PATCH_SIZE (smoke)  : {SMOKE_PATCH_SIZE}")
print(f"EPOCHS              : {EPOCHS}")
print(f"BATCH_SIZE          : {BATCH_SIZE}")
print()
print("VALIDATION SCHEDULE (formal only)")
print(f"  validation cases    : 306 (all)")
print(f"  formal overlap      : {VAL_OVERLAP}")
print(f"  validation frequency: every {VAL_EVERY} epochs")
print(f"  sw_batch_size       : {VAL_SW_BATCH_SIZE}")
print(f"  checkpoint_every    : {CHECKPOINT_EVERY}")
print()
print(f"k_att T             : {K_ATT_TEMP_START} -> {K_ATT_TEMP_END} over {K_ATT_ANNEAL_EPOCHS} ep")
print(f"attn_diversity_wt   : {ATTN_DIVERSITY_WEIGHT}  (disabled)")
print(f"RESUME_FROM         : {RESUME_FROM or '(none — fresh training)'}")
print()
print("WARNING: FORMAL validation over all 306 cases at overlap=0.5 may take")
print("         SEVERAL HOURS on Kaggle GPUs. Plan runs accordingly.")


In [ ]:
# ── 1. INSTALL DEPS + REQUIRE CUDA ─────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "monai",
                "--upgrade-strategy", "only-if-needed", "-q"], check=True)

import torch
print(f"PyTorch        : {torch.__version__}")
print(f"CUDA available : {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise SystemExit(
        "CUDA is not available on this session. This notebook is designed "
        "to run on a GPU-backed Kaggle kernel — switch the accelerator "
        "to GPU (any single-GPU option works) and re-run.\n"
        "NOTE: This training code is SINGLE-GPU and uses only cuda:0. "
        "Selecting the 'T4 x2' accelerator does NOT combine both GPUs or "
        "double the available VRAM — the second T4 sits idle. Use 'P100' "
        "or 'T4 x2' interchangeably; only cuda:0's VRAM budget matters."
    )
print(f"GPU (cuda:0)   : {torch.cuda.get_device_name(0)}")
p = torch.cuda.get_device_properties(0)
print(f"VRAM (cuda:0)  : {p.total_memory / 1e9:.1f} GB")
print("NOTE: this notebook uses only cuda:0. 'T4 x2' is NOT multi-GPU here.")


In [ ]:
# ── 2. CLONE / CHECKOUT feature/fadc3d-correct (subprocess, check=True) ─
import os, sys, subprocess

def _run(cmd, cwd=None):
    """subprocess.run wrapper that surfaces stderr and check=True."""
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:
        sys.stdout.write(r.stdout)
        sys.stderr.write(r.stderr)
        raise SystemExit(f"Command failed ({r.returncode}): {' '.join(cmd)}")
    return r.stdout.strip()

if os.path.exists(CODE_DIR):
    print(f"repo present; fetching {GIT_BRANCH} ...")
    _run(["git", "-C", CODE_DIR, "fetch", "--all"])
    _run(["git", "-C", CODE_DIR, "checkout", GIT_BRANCH])
    _run(["git", "-C", CODE_DIR, "pull", "--ff-only"])
else:
    _run(["git", "clone", "-b", GIT_BRANCH,
          "https://github.com/Vemuri-BK/FADC-3D.git", CODE_DIR])

sys.path.insert(0, CODE_DIR)

# Verify active branch matches GIT_BRANCH and capture the exact commit hash.
active = _run(["git", "-C", CODE_DIR, "rev-parse", "--abbrev-ref", "HEAD"])
if active != GIT_BRANCH:
    raise SystemExit(f"Active branch is {active!r}, expected {GIT_BRANCH!r}. "
                     "Refusing to run on the wrong branch.")
GIT_COMMIT_HASH = _run(["git", "-C", CODE_DIR, "rev-parse", "HEAD"])
GIT_COMMIT_LINE = _run(["git", "-C", CODE_DIR, "log", "-1", "--oneline"])
print(f"branch  : {active}")
print(f"HEAD    : {GIT_COMMIT_LINE}")
print(f"commit  : {GIT_COMMIT_HASH}")

# Persist the commit hash next to full-training outputs so a downloaded
# checkpoint can be traced back to its source commit later.
os.makedirs(OUTPUT_DIR_FULL, exist_ok=True)
with open(os.path.join(OUTPUT_DIR_FULL, "source_commit.txt"), "w", encoding="utf-8") as f:
    f.write(GIT_COMMIT_HASH + "\n" + GIT_COMMIT_LINE + "\n")

for p in ("fadc_3d_correct/adaptive_dilated_conv_3d.py",
          "fadc_3d_correct/ada_kernel_3d.py",
          "fadc_3d_correct/freq_select_3d.py",
          "models/unet_3d_fadc_correct.py",
          "training/train_centralized_correct.py",
          "tests/test_fadc_3d_correct.py",
          "diag_fadc_3d_correct.py"):
    assert os.path.exists(os.path.join(CODE_DIR, p)), f"missing on branch: {p}"
print("Corrected FADC3D files present on branch.")


In [ ]:
# ── 3. PULL VERIFIER + ARCHITECTURE CHECKS ─────────────────────────────
# Aborts BEFORE training if the corrected code did not land. Releases every
# temporary tensor / module before returning control to later cells.
import gc, inspect, sys, torch
sys.path.insert(0, CODE_DIR)

from fadc_3d_correct.adaptive_dilated_conv_3d import AdaptiveDilatedConv3D
from fadc_3d_correct.ada_kernel_3d import AdaKern3D
from fadc_3d_correct.freq_select_3d import FrequencySelection3D
from models.unet_3d_fadc_correct import (
    build_unet3d_fadc_correct, EXPECTED_ADAPTIVE_CONV_COUNT, MODEL_NAMES,
)

# 1) module signatures
assert AdaptiveDilatedConv3D.KERNEL_SIZE == 3
assert MODEL_NAME in MODEL_NAMES, f"MODEL_NAME {MODEL_NAME} not in {MODEL_NAMES}"

# 2) build fresh encoder-only model
m = build_unet3d_fadc_correct(MODEL_NAME, in_channels=2, out_channels=2,
                              base_filters=32, deep_supervision=DEEP_SUPERVISION).cuda()

# 3) EXACTLY 8 adaptive convs, all under enc*
n_adapt = m.count_adaptive_convs()
assert n_adapt == EXPECTED_ADAPTIVE_CONV_COUNT["encoder"] == 8, \
    f"encoder placement must yield 8 adaptive convs, got {n_adapt}"
adapt_names = m.adaptive_conv_names()
outside_enc = [n for n in adapt_names if not n.startswith("enc")]
assert not outside_enc, f"adaptive convs found outside enc*: {outside_enc}"

# 4) sanity: single base kernel per adaptive conv
for name, mod in m.named_modules():
    if isinstance(mod, AdaptiveDilatedConv3D):
        top_weights = [n for n, p in mod.named_parameters(recurse=False) if n == "weight"]
        assert top_weights == ["weight"], f"{name} has unexpected top-level kernel params: {top_weights}"

# 5) dilation list is (1, 2, 3)
sample = next(mm for mm in m.modules() if isinstance(mm, AdaptiveDilatedConv3D))
assert sample.dilation_list == (1, 2, 3), sample.dilation_list

print(f"MODEL_NAME       : {MODEL_NAME}")
print(f"adaptive convs   : {n_adapt}/8 (all under enc*)")
print(f"params           : {sum(p.numel() for p in m.parameters()):,}")
print("Pull + architecture checks passed.")

# Free the check-only model + the iterator ref before later cells run.
del sample, m
gc.collect()
torch.cuda.empty_cache()
print(f"post-check VRAM alloc: {torch.cuda.memory_allocated()/1e9:.2f} GB")


In [ ]:
# ── 4. RUN CORRECTNESS TESTS (abort on nonzero) ───────────────────────
# Invoke each test file by ABSOLUTE PATH — avoids depending on `tests/`
# being an importable package (no tests/__init__.py). Each test module
# adds the repo root to sys.path itself, so this Just Works.
import os, subprocess, sys

TEST_FILES = [
    "tests/test_fadc_3d_correct.py",       # FADC3D math correctness (35+ assertions)
    "tests/test_train_correct_utils.py",   # training loop utilities (formal-only path)
]

for rel in TEST_FILES:
    test_path = os.path.join(CODE_DIR, rel)
    assert os.path.exists(test_path), f"test file missing: {test_path}"
    print(f"---- running {rel} ----")
    res = subprocess.run(
        [sys.executable, test_path],
        cwd=CODE_DIR, capture_output=True, text=True,
    )
    print(res.stdout[-6000:])
    if res.returncode != 0:
        sys.stderr.write(res.stderr[-3000:])
        raise SystemExit(f"Test file {rel} FAILED (exit {res.returncode}) — refusing to launch training.")
    print(f"---- {rel} PASSED ----\n")
print("All correctness tests PASSED.")


In [ ]:
# ── 5. SMOKE TRAINING — 2 ep on 4 cases, small patch (abort on nonzero) ─
import os, subprocess, sys

train_script = os.path.join(CODE_DIR, "training", "train_centralized_correct.py")
os.makedirs(OUTPUT_DIR_SMOKE, exist_ok=True)

cmd = [
    sys.executable, "-u", train_script,
    "--model",         MODEL_NAME,
    "--data_root",     DATA_ROOT,
    "--output_dir",    OUTPUT_DIR_SMOKE,
    "--patch_size",    str(SMOKE_PATCH_SIZE[0]), str(SMOKE_PATCH_SIZE[1]), str(SMOKE_PATCH_SIZE[2]),
    "--batch_size",    "2",
    "--warmup_epochs", "1",
    "--val_every",     "1",
    "--val_overlap",   str(VAL_OVERLAP),
    "--val_sw_batch_size", str(VAL_SW_BATCH_SIZE),
    "--seed",          str(SEED),
    "--k_att_temp_start", str(K_ATT_TEMP_START),
    "--k_att_temp_end",   str(K_ATT_TEMP_END),
    "--k_att_anneal_epochs", "1",
    "--preprocessed_cache_dir", PREPROCESSED_CACHE_DIR,
    "--smoke_test",
]
if DEEP_SUPERVISION:
    cmd.append("--deep_supervision")

print("SMOKE command:\n  " + " ".join(cmd))
print("=" * 60, flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
while True:
    chunk = proc.stdout.read(512)
    if not chunk: break
    sys.stdout.write(chunk.decode("utf-8", errors="replace"))
    sys.stdout.flush()
proc.wait()
print(f"\nSMOKE exit code: {proc.returncode}")
if proc.returncode != 0:
    raise SystemExit(f"SMOKE training failed (exit {proc.returncode}). Refusing to launch full training.")


In [ ]:
# ── 6. SMOKE CKPT: strict=True reload + forward parity ────────────────
import os, torch, sys
sys.path.insert(0, CODE_DIR)
from models.unet_3d_fadc_correct import build_unet3d_fadc_correct

ckpt_path = os.path.join(OUTPUT_DIR_SMOKE, "last_checkpoint.pth")
assert os.path.exists(ckpt_path), f"smoke checkpoint missing: {ckpt_path}"
ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)
print(f"epoch    : {ckpt.get('epoch')}")
print(f"best_dice: {ckpt.get('best_dice')}")
arch = ckpt.get("arch_identity")
print(f"arch_id  : {arch}")

model = build_unet3d_fadc_correct(
    arch["model_name"],
    in_channels=arch["in_channels"], out_channels=arch["out_channels"],
    base_filters=arch["base_filters"], deep_supervision=arch["deep_supervision"],
).eval()
missing, unexpected = model.load_state_dict(ckpt["model"], strict=True)
print(f"strict load: missing={len(missing)} unexpected={len(unexpected)}")
assert not missing and not unexpected, "strict reload mismatch"

x = torch.randn(1, 2, 32, 32, 16)
with torch.no_grad():
    y1 = model(x)
    y2 = model(x)
diff = (y1 - y2).abs().max().item()
assert diff < 1e-6, f"non-deterministic forward under eval(): {diff}"
print(f"forward parity  max|y1-y2| = {diff:.2e}  OK")


In [ ]:
# ── 7. GPU MEMORY PROBE — one 128x128x64 batch, INCLUDING every DS head ─
# If this OOMs we STOP. We do NOT silently shrink the patch or batch size.
#
# The probe is wrapped in run_gpu_memory_probe() so every local (model, scaler,
# x, outputs, loss, autograd graph, gradients) falls out of scope on return.
# Followed by gc.collect + torch.cuda.empty_cache + synchronize so the full-
# training subprocess starts with a clean allocator, not with the probe's
# ~backward graph still resident.
import torch, gc, os, sys
sys.path.insert(0, CODE_DIR)
from models.unet_3d_fadc_correct import build_unet3d_fadc_correct
from torch.amp import autocast, GradScaler


def run_gpu_memory_probe() -> float:
    """Full-size forward + backward + unscaled step. Returns peak alloc (GB).

    Every tensor and module created here becomes unreachable once the function
    returns — so the caller only has to call gc.collect + empty_cache to
    reclaim VRAM. Nothing important is passed out.
    """
    torch.cuda.empty_cache()
    gc.collect()
    torch.cuda.reset_peak_memory_stats()
    print(f"pre-probe VRAM alloc: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    model = build_unet3d_fadc_correct(
        MODEL_NAME, in_channels=2, out_channels=2, base_filters=32,
        deep_supervision=DEEP_SUPERVISION,
    ).cuda().train()

    scaler = GradScaler("cuda")
    x = torch.randn(BATCH_SIZE, 2, *PATCH_SIZE, device="cuda")
    with autocast("cuda"):
        y = model(x)
        # y is (main, ds2, ds3, ds4) under deep supervision; include EVERY
        # output so gradients flow through the auxiliary heads too.
        outputs = y if isinstance(y, tuple) else (y,)
        loss = sum(o.float().pow(2).mean() for o in outputs)
    scaler.scale(loss).backward()

    peak = torch.cuda.max_memory_allocated() / 1e9
    print(f"probe OK. n_outputs_in_loss={len(outputs)} "
          f"peak VRAM alloc: {peak:.2f} GB")
    return peak


try:
    _peak = run_gpu_memory_probe()
except RuntimeError as e:
    if "out of memory" in str(e).lower():
        _peak = torch.cuda.max_memory_allocated() / 1e9
        print(f"OOM at probe. peak alloc: {_peak:.2f} GB")
        # Still clean up before raising SystemExit so the notebook state
        # remains observable.
        gc.collect(); torch.cuda.empty_cache(); torch.cuda.synchronize()
        raise SystemExit(
            "Full-size probe OOMed. Not launching full training. "
            "Do NOT silently change patch_size or batch_size — the "
            "experiment contract requires the declared PATCH_SIZE/BATCH_SIZE."
        )
    raise

# Post-probe cleanup: everything inside run_gpu_memory_probe() is now
# unreachable; drop refcounts and reclaim VRAM before the full-training
# subprocess launches.
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

print(f"post-cleanup memory_allocated : {torch.cuda.memory_allocated()/1e9:.3f} GB")
print(f"post-cleanup memory_reserved  : {torch.cuda.memory_reserved()/1e9:.3f} GB")


In [ ]:
# ── 8. FULL TRAINING — 100 ep (formal validation only; abort on nonzero) ─
# If RESUME_FROM is non-empty the training script picks up from that
# checkpoint. The script's _require_matching_arch refuses to load a
# checkpoint with a different arch_identity.
import os, subprocess, sys

train_script = os.path.join(CODE_DIR, "training", "train_centralized_correct.py")
os.makedirs(OUTPUT_DIR_FULL, exist_ok=True)

cmd = [
    sys.executable, "-u", train_script,
    "--model",              MODEL_NAME,
    "--data_root",          DATA_ROOT,
    "--output_dir",         OUTPUT_DIR_FULL,
    "--epochs",             str(EPOCHS),
    "--batch_size",         str(BATCH_SIZE),
    "--num_workers",        str(NUM_WORKERS),
    "--patch_size",         str(PATCH_SIZE[0]), str(PATCH_SIZE[1]), str(PATCH_SIZE[2]),
    "--lr",                 str(LEARNING_RATE),
    "--warmup_epochs",      str(WARMUP_EPOCHS),
    "--seed",               str(SEED),
    "--k_att_temp_start",   str(K_ATT_TEMP_START),
    "--k_att_temp_end",     str(K_ATT_TEMP_END),
    "--k_att_anneal_epochs",str(K_ATT_ANNEAL_EPOCHS),
    "--preprocessed_cache_dir", PREPROCESSED_CACHE_DIR,
    # Formal validation only.
    "--val_every",          str(VAL_EVERY),
    "--val_overlap",        str(VAL_OVERLAP),
    "--val_sw_batch_size",  str(VAL_SW_BATCH_SIZE),
    "--checkpoint_every",   str(CHECKPOINT_EVERY),
]
if DEEP_SUPERVISION:
    cmd.append("--deep_supervision")

if RESUME_FROM:
    if not os.path.exists(RESUME_FROM):
        raise SystemExit(f"RESUME_FROM is set but the file does not exist: {RESUME_FROM}")
    cmd += ["--resume", RESUME_FROM]
    print(f"RESUME mode: continuing from {RESUME_FROM}")
else:
    print("FRESH mode: starting from epoch 1 (no --resume passed)")

print("FULL command:\n  " + " ".join(cmd))
print("=" * 60, flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
while True:
    chunk = proc.stdout.read(512)
    if not chunk: break
    sys.stdout.write(chunk.decode("utf-8", errors="replace"))
    sys.stdout.flush()
proc.wait()
print(f"\nFULL exit code: {proc.returncode}")
if proc.returncode != 0:
    raise SystemExit(f"FULL training exited nonzero ({proc.returncode}). Downstream diagnostic skipped.")


In [ ]:
# ── 8b. EPOCH-40 FORMAL VALIDATION + POST-VAL CHECKPOINT RECOVERY ──────
# One-off recovery cell. Training epoch 40 completed but its formal
# validation was interrupted, so last_checkpoint.pth carries the ep40
# training state without any val_* metrics folded in, and no periodic
# checkpoint_epoch040.pth was written.
#
# This cell:
#   1) verifies /kaggle/working/outputs/fadc3d_correct_encoder_s42/last_checkpoint.pth
#      is the ep40 ckpt (ckpt["epoch"] + 1 == 40),
#   2) preserves a copy as checkpoint_epoch040.pth,
#   3) runs formal validation (all 306 cases, patch 128x128x64, overlap=0.5,
#      sw_batch_size=4, num_workers=4, primary segmentation output, tqdm),
#   4) writes formal_eval_ep040.json,
#   5) folds val_dice / val_iou / val_sensitivity / val_n_cases / val_overlap
#      into the ep40 train_log entry (or a minimal validation-only entry if
#      none exists — no fabricated loss values),
#   6) atomically re-saves the augmented checkpoint to BOTH
#      last_checkpoint.pth AND checkpoint_epoch040.pth (keeping
#      model / optimizer / scheduler / scaler / arch_identity / config /
#      train_log; ckpt["epoch"] stays 39 internally so resume advances to 40+1=41),
#   7) if ep40 Dice > previous best (0.6436), atomically overwrites
#      best_model.pth; otherwise leaves best_model.pth untouched and keeps
#      best_dice=0.6436 in the post-val ckpt.
#   8) atomically rewrites train_log.json.
#
# Aborts on any failure. Nothing here modifies model weights, FADC math,
# or training settings.
import os, sys, json, subprocess
sys.path.insert(0, CODE_DIR)
import torch
from training.train_centralized_correct import atomic_torch_save, atomic_json_write

CKPT_PATH   = os.path.join(OUTPUT_DIR_FULL, "last_checkpoint.pth")
EP40_PATH   = os.path.join(OUTPUT_DIR_FULL, "checkpoint_epoch040.pth")
BEST_PATH   = os.path.join(OUTPUT_DIR_FULL, "best_model.pth")
LOG_PATH    = os.path.join(OUTPUT_DIR_FULL, "train_log.json")
JSON_OUT    = os.path.join(OUTPUT_DIR_FULL, "formal_eval_ep040.json")
EVAL_SCRIPT = os.path.join(CODE_DIR, "training", "evaluate_correct_checkpoint.py")

TARGET_TRAINING_EPOCH = 40
EXPECTED_N_CASES      = 306
EXPECTED_OVERLAP      = 0.5
PREV_BEST_DICE        = 0.6436

# --- 1) load ckpt on CPU and verify epoch -----------------------------
if not os.path.exists(CKPT_PATH):
    raise SystemExit(f"No last_checkpoint.pth at {CKPT_PATH} — cannot proceed.")
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
ckpt_epoch = int(ckpt.get("epoch", -1))
completed_epoch = ckpt_epoch + 1
if completed_epoch != TARGET_TRAINING_EPOCH:
    raise SystemExit(
        f"Refusing to run epoch-{TARGET_TRAINING_EPOCH} validation: "
        f"last_checkpoint.pth reports completed training epoch {completed_epoch} "
        f"(ckpt.epoch={ckpt_epoch}), not {TARGET_TRAINING_EPOCH}."
    )
print(f"OK: last_checkpoint.pth is training epoch {completed_epoch} (ckpt.epoch={ckpt_epoch}).")
print(f"    arch_identity            : {ckpt.get('arch_identity')}")
print(f"    ckpt best_dice (before)  : {float(ckpt.get('best_dice', 0.0)):.4f}")
print(f"    previous formal best     : {PREV_BEST_DICE:.4f}")

# --- 2) preserve a safety copy at checkpoint_epoch040.pth -------------
# The periodic snapshot may not exist yet (validation was interrupted before
# that write). Save a copy of the pre-val ckpt now, then overwrite it with
# the val-augmented version after step 6.
atomic_torch_save(ckpt, EP40_PATH)
print(f"pre-val safety copy -> {EP40_PATH}")

# --- 3) run formal validation subprocess ------------------------------
cmd = [
    sys.executable, "-u", EVAL_SCRIPT,
    "--checkpoint",             CKPT_PATH,
    "--data_root",              DATA_ROOT,
    "--preprocessed_cache_dir", PREPROCESSED_CACHE_DIR,
    "--patch_size",             str(PATCH_SIZE[0]), str(PATCH_SIZE[1]), str(PATCH_SIZE[2]),
    "--overlap",                str(EXPECTED_OVERLAP),
    "--sw_batch_size",          str(VAL_SW_BATCH_SIZE),
    "--num_workers",            str(NUM_WORKERS),
    "--out",                    JSON_OUT,
]
print("FORMAL eval command:\n  " + " ".join(cmd))
print("=" * 60, flush=True)
proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
while True:
    chunk = proc.stdout.read(512)
    if not chunk: break
    sys.stdout.write(chunk.decode("utf-8", errors="replace"))
    sys.stdout.flush()
proc.wait()
if proc.returncode != 0:
    raise SystemExit(f"Formal eval subprocess exited nonzero ({proc.returncode}). "
                     "Refusing to fold results into the checkpoint.")

# --- 4) parse + validate JSON output ----------------------------------
if not os.path.exists(JSON_OUT):
    raise SystemExit(f"Formal eval subprocess reported success but {JSON_OUT} is missing.")
with open(JSON_OUT, encoding="utf-8") as f:
    eval_result = json.load(f)

def _require(cond, msg):
    if not cond:
        raise SystemExit(f"Post-val guard failed: {msg}")

_require(int(eval_result.get("n_cases", -1)) == EXPECTED_N_CASES,
         f"n_cases={eval_result.get('n_cases')} != {EXPECTED_N_CASES}")
_require(abs(float(eval_result.get("overlap", -1.0)) - EXPECTED_OVERLAP) < 1e-9,
         f"overlap={eval_result.get('overlap')} != {EXPECTED_OVERLAP}")
_require(int(eval_result.get("checkpoint_epoch", -999)) + 1 == TARGET_TRAINING_EPOCH,
         f"evaluated ckpt training epoch {int(eval_result.get('checkpoint_epoch', -999)) + 1} "
         f"!= {TARGET_TRAINING_EPOCH}")

new_dice = float(eval_result["dice"])
new_iou  = float(eval_result["iou"])
new_sens = float(eval_result["sensitivity"])

# --- 5) augment train_log ep40 entry ---------------------------------
train_log = ckpt.get("train_log", [])
if not isinstance(train_log, list):
    train_log = []
ep40_idx = [i for i, e in enumerate(train_log)
            if isinstance(e, dict) and int(e.get("epoch", -1)) == TARGET_TRAINING_EPOCH]
val_fields = {
    "val_dice":        new_dice,
    "val_iou":         new_iou,
    "val_sensitivity": new_sens,
    "val_n_cases":     EXPECTED_N_CASES,
    "val_overlap":     EXPECTED_OVERLAP,
}
if ep40_idx:
    idx = ep40_idx[-1]
    train_log[idx].update(val_fields)
    print(f"train_log[{idx}] (epoch {TARGET_TRAINING_EPOCH}) augmented with val_* fields")
else:
    # No epoch-40 log entry — add a minimal validation-only record.
    # DO NOT fabricate loss / dice_loss / ce_loss values.
    train_log.append({"epoch": TARGET_TRAINING_EPOCH, **val_fields})
    print(f"no existing epoch-{TARGET_TRAINING_EPOCH} log entry — appended a validation-only minimal entry")
ckpt["train_log"] = train_log

# --- 6) best-dice decision ------------------------------------------------
became_best = new_dice > PREV_BEST_DICE
if became_best:
    ckpt["best_dice"] = new_dice
else:
    # Explicitly pin to the known previous best regardless of what the
    # loaded ckpt happened to carry, so the post-val checkpoint is unambiguous.
    ckpt["best_dice"] = PREV_BEST_DICE

# --- 7) atomically save augmented ckpt to last_checkpoint + ep040 --------
atomic_torch_save(ckpt, CKPT_PATH)
atomic_torch_save(ckpt, EP40_PATH)
print(f"post-val ckpt (augmented) -> {CKPT_PATH}")
print(f"post-val ckpt (augmented) -> {EP40_PATH}")

# --- 8) update best_model.pth conditionally -------------------------------
if became_best:
    atomic_torch_save(ckpt, BEST_PATH)
    print(f"NEW FORMAL BEST: {new_dice:.4f} > {PREV_BEST_DICE:.4f} -> best_model.pth updated")
else:
    print(f"previous formal best preserved: {PREV_BEST_DICE:.4f} >= {new_dice:.4f}; "
          f"best_model.pth UNCHANGED (still epoch-20 checkpoint).")

# --- 9) rewrite train_log.json atomically ---------------------------------
atomic_json_write(train_log, LOG_PATH)
print(f"train_log.json rewritten -> {LOG_PATH}")

# --- 10) final summary ---------------------------------------------------
print()
print("=" * 60)
print("EPOCH-40 POST-VAL RECOVERY COMPLETE")
print(f"  epoch evaluated          : {TARGET_TRAINING_EPOCH}")
print(f"  FORMAL Dice              : {new_dice:.4f}")
print(f"  FORMAL IoU               : {new_iou:.4f}")
print(f"  FORMAL Sensitivity       : {new_sens:.4f}")
print(f"  n_cases / overlap        : {EXPECTED_N_CASES} / {EXPECTED_OVERLAP}")
print(f"  previous best Dice       : {PREV_BEST_DICE:.4f}")
print(f"  epoch 40 became new best : {'YES' if became_best else 'no'}")
print(f"  best_dice in post-val ck : {float(ckpt['best_dice']):.4f}")
print(f"  last_checkpoint.pth      : {CKPT_PATH}")
print(f"  checkpoint_epoch040.pth  : {EP40_PATH}")
print(f"  best_model.pth           : {BEST_PATH}")
print(f"  formal_eval_ep040.json   : {JSON_OUT}")
print()
print(f"Training can now resume from last_checkpoint.pth "
      f"(ckpt.epoch={ckpt_epoch}, next training epoch = {TARGET_TRAINING_EPOCH + 1}).")


In [ ]:
# ── 8c. EPOCH-N FORMAL EVAL (standalone; run before resuming training) ─
# Runs formal validation (all cases, overlap=0.5) on a specific saved
# periodic checkpoint — typically checkpoint_epoch020.pth after the first
# 20-epoch training window. Reports the checkpoint's recorded epoch and
# the git commit written next to OUTPUT_DIR_FULL. Never touches the
# checkpoint. Writes the metrics to a JSON file next to the checkpoint.
#
# Comparison target: existing overlap=0.5 formal Dice at epoch 10 = 0.5429.
# The number this cell reports IS directly comparable to that.
# It is NOT comparable to the deprecated fast/proxy Dice (e.g. 0.45).
import os, subprocess, sys

# Which periodic checkpoint to evaluate.
EP20_CHECKPOINT = os.path.join(OUTPUT_DIR_FULL, "checkpoint_epoch020.pth")

if not os.path.exists(EP20_CHECKPOINT):
    print(f"no checkpoint at {EP20_CHECKPOINT}; skipping epoch-N formal eval.")
    print("Available checkpoints in OUTPUT_DIR_FULL:")
    if os.path.isdir(OUTPUT_DIR_FULL):
        for name in sorted(os.listdir(OUTPUT_DIR_FULL)):
            if name.endswith(".pth"):
                print(f"  {name}")
else:
    # Print checkpoint identity BEFORE evaluating so the reader can trace it.
    import torch
    _ck = torch.load(EP20_CHECKPOINT, map_location="cpu", weights_only=False)
    _epoch = int(_ck.get("epoch", -1))
    _best_selfrep = float(_ck.get("best_dice", 0.0))
    _arch = _ck.get("arch_identity", {})
    del _ck
    _commit_file = os.path.join(OUTPUT_DIR_FULL, "source_commit.txt")
    _commit = "(unknown)"
    if os.path.exists(_commit_file):
        with open(_commit_file, encoding="utf-8") as fh:
            _commit = fh.read().strip().splitlines()[0] if fh else "(unknown)"
    print(f"epoch-N formal eval target")
    print(f"  checkpoint             : {EP20_CHECKPOINT}")
    print(f"  checkpoint epoch (0idx): {_epoch}   (means training epoch {_epoch+1} completed)")
    print(f"  best_dice self-reported: {_best_selfrep:.4f}")
    print(f"  git commit             : {_commit}")
    print(f"  arch_identity          : {_arch}")

    eval_script = os.path.join(CODE_DIR, "training", "evaluate_correct_checkpoint.py")
    out_json = os.path.join(OUTPUT_DIR_FULL, f"formal_eval_ep{_epoch+1:03d}.json")
    cmd = [
        sys.executable, "-u", eval_script,
        "--checkpoint",             EP20_CHECKPOINT,
        "--data_root",              DATA_ROOT,
        "--preprocessed_cache_dir", PREPROCESSED_CACHE_DIR,
        "--patch_size",             str(PATCH_SIZE[0]), str(PATCH_SIZE[1]), str(PATCH_SIZE[2]),
        "--overlap",                str(VAL_OVERLAP),
        "--sw_batch_size",          str(VAL_SW_BATCH_SIZE),
        "--num_workers",            str(NUM_WORKERS),
        "--out",                    out_json,
    ]
    print("EPOCH-N FORMAL EVAL command:\n  " + " ".join(cmd))
    print("=" * 60, flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=0)
    while True:
        chunk = proc.stdout.read(512)
        if not chunk: break
        sys.stdout.write(chunk.decode("utf-8", errors="replace"))
        sys.stdout.flush()
    proc.wait()
    print(f"\nEPOCH-N eval exit code: {proc.returncode}")
    print(f"metrics written to: {out_json}")


In [ ]:
# ── 9. REAL-MRI DIAGNOSTIC ON best_model.pth ──────────────────────────
import os, subprocess, sys
best_ckpt = os.path.join(OUTPUT_DIR_FULL, "best_model.pth")
val_cache = os.path.join(PREPROCESSED_CACHE_DIR, "val")
if not os.path.exists(best_ckpt):
    print(f"no best checkpoint at {best_ckpt}; skipping diagnostic.")
else:
    cmd = [
        sys.executable, os.path.join(CODE_DIR, "diag_fadc_3d_correct.py"),
        "--ckpt", best_ckpt,
        "--preprocessed_cache", val_cache,
        "--n_patches", "4",
        "--patch_size", "96", "96", "48",
    ]
    print("DIAG command:\n  " + " ".join(cmd))
    print("=" * 60, flush=True)
    subprocess.run(cmd, check=False)


In [ ]:
# ── 10. DOWNLOAD LINKS ────────────────────────────────────────────────
import os
from IPython.display import FileLink, display
for fname in ("best_model.pth", "last_checkpoint.pth",
              "train_log.json", "meta.json", "source_commit.txt"):
    p = os.path.join(OUTPUT_DIR_FULL, fname)
    if os.path.exists(p):
        print(fname); display(FileLink(p))
    else:
        print(f"(missing) {fname}")
